# Surgical Phase Classification — End-to-End Colab Tutorial

This notebook is a **full runnable pipeline** for surgical phase classification using multiple methods:

- CNN + LSTM baseline
- Temporal ConvNet (TCN)
- Transformer temporal model
- Optional SimCLR pretraining scaffold

It also includes:
- dataset manifest creation
- train/val/test runs
- evaluation + confusion matrix + per-phase F1
- simple experiment comparison table

> Designed for medium Colab GPU sessions (T4/L4).


## 0) Runtime checklist

- Runtime type: **GPU**
- Python: 3.10+
- Expected folder after setup: `/content/surgical-phase-colab-starter`


In [ ]:
#@title Check GPU
!nvidia-smi


## 1) Clone repo and install dependencies

In [ ]:
#@title Clone + checkout feature branch
%cd /content
!rm -rf surgical-phase-colab-starter
!git clone https://github.com/ericnerwala/surgical-phase-colab-starter.git
%cd surgical-phase-colab-starter
!git checkout feat/modular-phase-pipeline


In [ ]:
#@title Install requirements
!pip install -q -r requirements.txt
!pip install -q -e .


## 2) Upload / mount dataset

Expected structure (example):

```
/content/data/cholect50-challenge-val/
  videos/
  labels/
```

If your data is on Drive, mount it. Otherwise upload/extract manually.


In [ ]:
#@title (Optional) Mount Google Drive
from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
#@title Set your dataset root
DATA_ROOT = '/content/data/cholect50-challenge-val'  # change if needed
print('DATA_ROOT =', DATA_ROOT)


## 3) Build manifest + phase map

This scans labels and frames, then creates:
- `data/processed/manifest.csv`
- `data/processed/phase_map.json`


In [ ]:
#@title Build manifest
!python scripts/build_manifest.py   --data-root "$DATA_ROOT"   --out-csv data/processed/manifest.csv   --phase-map-out data/processed/phase_map.json   --val-videos VID73   --test-videos VID75

!python - <<'PY2'
import pandas as pd
p='data/processed/manifest.csv'
df=pd.read_csv(p)
print('Rows:',len(df))
print(df.head())
print('Split counts:
',df['split'].value_counts(dropna=False))
PY2


## 4) Quick config knobs

In [ ]:
#@title Optional: patch configs for quick Colab runs
import yaml, pathlib

def patch_cfg(path, epochs=3, batch_size=8, num_workers=2):
    p=pathlib.Path(path)
    cfg=yaml.safe_load(p.read_text())
    cfg.setdefault('train', {})
    cfg['train']['epochs']=epochs
    cfg['train']['batch_size']=batch_size
    cfg['train']['num_workers']=num_workers
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))

for c in ['configs/base.yaml','configs/tcn.yaml','configs/transformer.yaml']:
    patch_cfg(c, epochs=3, batch_size=8, num_workers=2)
    print('Patched', c)


## 5) Train all three methods

In [ ]:
#@title Train CNN+LSTM
!python scripts/train.py --config configs/base.yaml


In [ ]:
#@title Train TCN
!python scripts/train.py --config configs/tcn.yaml


In [ ]:
#@title Train Transformer
!python scripts/train.py --config configs/transformer.yaml


## 6) Evaluate on test split

In [ ]:
#@title Evaluate CNN+LSTM
!python scripts/eval.py   --config configs/base.yaml   --checkpoint outputs/cnn_lstm_base/best.pt   --split test   --out outputs/cnn_lstm_base/test_metrics.json


In [ ]:
#@title Evaluate TCN
!python scripts/eval.py   --config configs/tcn.yaml   --checkpoint outputs/tcn_base/best.pt   --split test   --out outputs/tcn_base/test_metrics.json


In [ ]:
#@title Evaluate Transformer
!python scripts/eval.py   --config configs/transformer.yaml   --checkpoint outputs/transformer_base/best.pt   --split test   --out outputs/transformer_base/test_metrics.json


## 7) Compare results (accuracy, macro-F1)

In [ ]:
#@title Build comparison table
import json, pandas as pd
paths = {
  'cnn_lstm':'outputs/cnn_lstm_base/test_metrics.json',
  'tcn':'outputs/tcn_base/test_metrics.json',
  'transformer':'outputs/transformer_base/test_metrics.json',
}
rows=[]
for name,p in paths.items():
    with open(p) as f:
        m=json.load(f)
    rows.append({'model':name,'accuracy':m.get('accuracy'),'macro_f1':m.get('macro_f1')})

df=pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print(df.to_string(index=False))


## 8) Inspect per-phase F1 and confusion matrix

In [ ]:
#@title Show detailed metrics for best model
import json, pandas as pd
best_path='outputs/transformer_base/test_metrics.json'  # change if needed
with open(best_path) as f:
    m=json.load(f)

print('Accuracy:', m.get('accuracy'))
print('Macro-F1:', m.get('macro_f1'))

per_phase = m.get('per_phase_f1', {})
if per_phase:
    d = pd.DataFrame([{'phase':k,'f1':v} for k,v in per_phase.items()]).sort_values('f1')
    display(d)

cm = m.get('confusion_matrix')
if cm is not None:
    import matplotlib.pyplot as plt
    import numpy as np
    arr=np.array(cm)
    plt.figure(figsize=(6,5))
    plt.imshow(arr, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()
    plt.xlabel('Pred')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()


## 9) Optional: SimCLR pretraining scaffold

In [ ]:
#@title Optional SimCLR pretraining
# !python scripts/pretrain_simclr.py --config configs/base.yaml --out outputs/simclr_pretrain.pt
print('Uncomment to run.')


## 10) Save artifacts

In [ ]:
#@title Zip outputs
!zip -r /content/surgical_phase_outputs.zip outputs
print('Saved /content/surgical_phase_outputs.zip')


## Troubleshooting

- OOM? Reduce batch size and sequence length in configs.
- Dataloader hangs? Set `num_workers: 0-2`.
- Bad convergence? Increase epochs and add class-balanced sampling / boundary-aware loss next.
